# <center>TensorRT Multiple Execution Contexts</center>

TensorRT 作为一款强大的深度学习推理优化工具，已经在模型推理速度上实现了显著提升。然而，如果能够进一步基于 TensorRT 实现并行推理，不仅可以有效降低推理延迟，还能显著提升服务吞吐量，这无疑是极具吸引力的优化方向。

在 TensorRT 的官方文档 [How TensorRT Works - Threading](https://docs.nvidia.com/deeplearning/tensorrt/latest/architecture/how-trt-works.html#threading) 中，我们了解到 <b>TensorRT 并非线程安全的，需要开发者自行管理不同线程之间的对象访问。理想的运行时并发模式是让不同的线程操作不同的 TensorRT 执行上下文（ExecutionContext），如果在不同线程中使用同一个执行上下文，将会导致未定义的行为。</b> 此外，从 TensorRT 的官方文档 [C++ API Documentation - Performing Inference](https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/c-api-docs.html#performing-inference) 中，我们还了解到<b>可以从一个反序列化引擎构建多个执行上下文，并且允许一个权重用于多个重叠的推理任务。</b>

## <center>TensorRT 多线程并发推理方案</center>

基于官方文档的信息，若需在 TensorRT 中实现多线程并发推理，可考虑以下几种方案：

### 1. 多引擎模式（Multiple Engines）

在多引擎模式下，我们创建一个线程池，并在其中启动多个工作线程。每个工作线程独立加载模型、创建引擎，并基于该引擎生成执行上下文。随后，各线程使用各自的上下文进行推理任务。

然而，这种模式存在一定的资源调度限制。CPU 端的计算资源划分单位是进程，而 GPU 端的计算资源调度单位是 context。在同一时刻，GPU 仅允许一个 context 执行（未开启 [MPS](https://docs.nvidia.com/deploy/mps/index.html#why-mps-is-needed) 的情况下）。这意味着，尽管多个 CPU 进程可以并行执行，但 GPU 在微观层面上会采用时间片轮转调度方式，多个任务交替运行。因此，即使在每个线程中创建独立的上下文，当多个线程同时执行推理任务时，这些任务在 GPU 端仍然是排队执行的。

### 2. 多上下文模式（Multiple Contexts）

在多上下文模式下，我们在主线程中统一加载模型文件并创建引擎。然后启动线程池，创建多个工作线程。每个工作线程基于主线程中的引擎对象，生成独立的执行上下文。各线程使用各自的上下文执行推理操作。

这种模式适合小数据量或低延迟需求的任务，能够通过多线程并发提升吞吐量，同时避免了引擎创建的开销。

### 3. 单引擎单上下文模式

在这种模式下，我们创建一个引擎，并基于该引擎生成执行上下文。然后在线程中收集数据后，使用该上下文进行推理。推理完成后，将结果分发至相应线程。

这种模式适合大数据量的任务，能够通过集中管理和批处理最大化 GPU 的计算资源利用率。

综合来看，真正可行的方案是**多上下文模式**和**单引擎单上下文模式**。其中，**多上下文模式**适合小数据量或低延迟需求的任务，能够通过多线程并发提升吞吐量，同时避免引擎创建的开销；而**单引擎单上下文模式**适合大数据量的任务，能够通过集中管理和批处理最大化 GPU 的计算资源利用率。此外，这两种方案也可以结合使用，以最大化服务的吞吐量。

## <center>多上下文模式（Multiple Contexts）的实现</center>

接下来我们将具体实现如何在一个推理引擎中构建多个上下文（Context）。与之前的流程类似，但关键区别在于，我们不再需要创建反序列化引擎，而是通过共享引擎指针来实现。具体来说，我们将创建新的上下文，并完成 I/O 绑定以及相关的初始化工作。需要注意的是，释放资源的顺序很重要：我们必须先释放所创建的上下文，然后才能释放引擎和运行时（Runtime）。

在上一章中，我们设计了一个 `TRTManager` 类，用于在全局范围内共享同一个运行时（Runtime），并为不同的模型创建独立的引擎（Engine）和上下文（Context）。当需要实现多上下文（Multiple Contexts）模式时，我们只需为 `InferBackend` 类添加一个 `clone` 方法，具体实现如下：


```cpp
std::unique_ptr<InferBackend> InferBackend::clone() {
    auto clone_backend = std::make_unique<InferBackend>(); // 创建一个新的InferBackend实例

    cudaSetDevice(0); // 设置CUDA设备为设备0
    CHECK(cudaStreamCreate(&clone_backend->stream)); // 创建一个CUDA流

    clone_backend->engine_ = engine_; // 共享当前引擎指针
    clone_backend->context_ = std::unique_ptr<nvinfer1::IExecutionContext>(
        clone_backend->engine_->createExecutionContext()); // 为共享引擎创建一个新的上下文
    if (!clone_backend->context_) {
        throw std::runtime_error("Failed to call createExecutionContext()."); // 如果创建失败，抛出异常
    }

    // 获取Tensor信息
    clone_backend->getTensorInfo();

    // 初始化相关变量
    clone_backend->initialize();

    return clone_backend; // 返回克隆后的InferBackend实例
}
```

从代码中可以看出，`clone` 方法与 `InferBackend` 的构造函数非常相似，但核心区别在于它共享了引擎指针，而不是重新创建引擎。通过这种方式，我们能够高效地为同一个模型创建多个上下文，从而实现多上下文推理模式。

尽管全局范围内共享同一个运行时（Runtime）能够在一定程度上减少初始化开销，但在创建多个带有自定义插件的反序列化引擎时，可能会遇到类似以下错误：

```shell
IRuntime::deserializeCudaEngine: Error Code 4: API Usage Error (Cannot register the library as plugin creator of EfficientRotatedNMS_TRT exists already.)
```

该错误表明，当多个引擎尝试注册同一个自定义插件时，运行时会报错，因为插件创建器已被注册，无法重复注册。为解决这一问题，主要有以下两种方法：

1. **不将插件序列化进引擎**  
   在构建引擎时，避免使用 `--setPluginsToSerialize` 选项将插件序列化到引擎中。相反，在 `TRTManager` 中（反序列化引擎之前）全局动态加载插件库（`LoadLibraryA` 或 `dlopen`）。这样可以确保插件在运行时被正确注册，而不会因重复注册导致冲突。

2. **创建独立的运行时**  
   为每个 `InferBackend` 创建独立的运行时实例。通过这种方式，每个运行时可以独立管理其插件注册，从而避免插件创建器冲突的问题。虽然这种方法会增加运行时的初始化开销，但它能有效解决多模型场景下插件注册冲突的问题，同时提高系统的灵活性和可扩展性。

为了更好地解决这些问题，我们选择第二种方法，对 `TRTManager` 进行重构，使其能够管理 `InferBackend` 中的运行时（Runtime）、引擎（Engine）和上下文（Context）。以下是重构后的代码实现：

```cpp
/**
 * @class TRTManager
 * @brief TensorRT 管理器类，用于管理 TensorRT 的上下文、引擎和运行时。
 * 提供初始化、克隆、设置张量地址、设置输入形状、执行推理等方法。
 */
class TRTManager {
public:
    /**
     * @brief 默认构造函数。
     */
    TRTManager() : context_(nullptr), engine_(nullptr), runtime_(nullptr), logger_(nullptr) {}

    /**
     * @brief 析构函数。
     */
    ~TRTManager() {
        context_.reset();
        engine_.reset();
        runtime_.reset();
        logger_.reset();
    }

    // 禁用拷贝和移动语义
    TRTManager(const TRTManager&)            = delete;  // < 禁用拷贝构造函数
    TRTManager& operator=(const TRTManager&) = delete;  // < 禁用拷贝赋值运算符
    TRTManager(TRTManager&&)                 = delete;  // < 禁用移动构造函数
    TRTManager& operator=(TRTManager&&)      = delete;  // < 禁用移动赋值运算符

    /**
     * @brief 初始化方法，用于加载 TensorRT 引擎。
     * @param blob 包含引擎数据的指针。
     * @param size 引擎数据的大小。
     */
    void initialize(void const* blob, std::size_t size) {
        logger_ = std::make_unique<TRTLogger>(nvinfer1::ILogger::Severity::kWARNING);

        initLibNvInferPlugins(logger_.get(), "");

        // 创建 TensorRT runtime
        runtime_ = std::unique_ptr<nvinfer1::IRuntime>(nvinfer1::createInferRuntime(*logger_));
        if (!runtime_) {
            throw std::runtime_error("Failed to create TensorRT runtime.");
        }

        // 反序列化引擎
        engine_ = std::shared_ptr<nvinfer1::ICudaEngine>(runtime_->deserializeCudaEngine(blob, size));
        if (!engine_) {
            throw std::runtime_error("Failed to deserialize CUDA engine.");
        }

        // 创建执行上下文
        context_ = std::unique_ptr<nvinfer1::IExecutionContext>(engine_->createExecutionContext());
        if (!context_) {
            throw std::runtime_error("Failed to create execution context.");
        }
    }

    /**
     * @brief 克隆方法，返回一个 TRTManager 的独占指针。
     * @return std::unique_ptr<TRTManager> 克隆的 TRTManager 对象。
     */
    std::unique_ptr<TRTManager> clone() const {
        if (!engine_ || !runtime_) {
            throw std::runtime_error("Invalid engine or runtime in TRTManager.");
        }

        // 创建新的 TRTManager 实例
        auto newManager = std::make_unique<TRTManager>();

        // 共享 engine
        newManager->engine_ = engine_;

        // 创建新的 context
        newManager->context_ = std::unique_ptr<nvinfer1::IExecutionContext>(newManager->engine_->createExecutionContext());
        if (!newManager->context_) {
            throw std::runtime_error("Failed to create new execution context during clone.");
        }

        return newManager;
    }

    /**
     * @brief 设置张量地址。
     * @param tensorName 张量名称。
     * @param data 张量数据指针。
     * @return bool 设置是否成功。
     */
    bool setTensorAddress(char const* tensorName, void* data) {
        return context_->setTensorAddress(tensorName, data);
    }

    /**
     * @brief 设置输入形状。
     * @param tensorName 张量名称。
     * @param dims 张量维度。
     * @return bool 设置是否成功。
     */
    bool setInputShape(char const* tensorName, nvinfer1::Dims const& dims) {
        return context_->setInputShape(tensorName, dims);
    }

    /**
     * @brief 在指定的 CUDA 流上执行推理。
     * @param stream CUDA 流。
     * @return bool 推理是否成功。
     */
    bool enqueueV3(cudaStream_t stream) {
        return context_->enqueueV3(stream);
    }

    /**
     * @brief 获取张量的形状。
     * @param tensorName 张量名称。
     * @return nvinfer1::Dims 张量的形状。
     */
    nvinfer1::Dims getTensorShape(char const* tensorName) const noexcept {
        return engine_->getTensorShape(tensorName);
    }

    /**
     * @brief 获取张量的数据类型。
     * @param tensorName 张量名称。
     * @return nvinfer1::DataType 张量的数据类型。
     */
    nvinfer1::DataType getTensorDataType(char const* tensorName) const noexcept {
        return engine_->getTensorDataType(tensorName);
    }

    /**
     * @brief 获取张量的输入输出模式。
     * @param tensorName 张量名称。
     * @return nvinfer1::TensorIOMode 张量的输入输出模式。
     */
    nvinfer1::TensorIOMode getTensorIOMode(char const* tensorName) const noexcept {
        return engine_->getTensorIOMode(tensorName);
    }

    /**
     * @brief 获取指定配置文件的张量形状。
     * @param tensorName 张量名称。
     * @param profileIndex 配置文件索引。
     * @param select 配置文件选择器。
     * @return nvinfer1::Dims 张量的形状。
     */
    nvinfer1::Dims getProfileShape(char const* tensorName, int32_t profileIndex, nvinfer1::OptProfileSelector select) const noexcept {
        return engine_->getProfileShape(tensorName, profileIndex, select);
    }

    /**
     * @brief 获取输入输出张量的数量。
     * @return int32_t 输入输出张量的数量。
     */
    int32_t getNbIOTensors() const noexcept {
        return engine_->getNbIOTensors();
    }

    /**
     * @brief 获取指定索引的输入输出张量名称。
     * @param index 张量索引。
     * @return char const* 张量名称。
     */
    char const* getIOTensorName(int32_t index) const noexcept {
        return engine_->getIOTensorName(index);
    }

private:
    std::unique_ptr<nvinfer1::IExecutionContext> context_;  // < TensorRT 执行上下文
    std::shared_ptr<nvinfer1::ICudaEngine>       engine_;   // < TensorRT CUDA 引擎
    std::unique_ptr<nvinfer1::IRuntime>          runtime_;  // < TensorRT 运行时
    std::unique_ptr<TRTLogger>                   logger_;   // < TensorRT 日志记录器
};
```

对应的 `InferBackend` 类的构造函数和 `clone` 方法也进行了调整，以适配新的 `TRTManager` 设计：

```cpp
InferBackend::InferBackend(const std::string& trt_engine_file) {
    cudaSetDevice(0);                  // 设置 CUDA 设备为设备 0
    CHECK(cudaStreamCreate(&stream));  // 创建 CUDA 流

    // 创建 TRTManager 实例
    manager_ = std::make_unique<TRTManager>();

    // 获取 Engine Buffer
    std::string engine_buffer;
    ReadBinaryFromFile(trt_engine_file, &engine_buffer);

    // 调用 initialize 方法进行初始化
    manager_->initialize(engine_buffer.data(), engine_buffer.size());

    // 获取 TensorInfo
    getTensorInfo();

    // 初始化相关变量
    initialize();
}

std::unique_ptr<InferBackend> InferBackend::clone() {
    auto clone_backend = std::make_unique<InferBackend>(); // 创建一个新的InferBackend实例

    cudaSetDevice(0); // 设置CUDA设备为设备0
    CHECK(cudaStreamCreate(&clone_backend->stream)); // 创建一个CUDA流

    clone_backend->manager_ = manager_->clone();

    // 获取Tensor信息
    clone_backend->getTensorInfo();

    // 初始化相关变量
    clone_backend->initialize();

    return clone_backend; // 返回克隆后的InferBackend实例
}
```

## <center>总结</center>

本节课程深入探讨了 TensorRT 的多线程并发推理机制。我们学习了多引擎模式、多上下文模式和单引擎单上下文模式的优缺点，并重点实现了多上下文模式，通过共享引擎和创建独立执行上下文，显著提升了推理吞吐量和效率。同时，我们解决了自定义插件的注册冲突问题，增强了系统的灵活性和可扩展性。这些内容为模型的高效部署和推理优化提供了重要支持。